# Investigating `fsl_val_rl`: locating the induction-head phase change

This notebook reads the `log.h5` that `main.py` writes for each run and finds where the induction-head phase change happens on the `fsl_val_rl` evaluator (held-out label pairs, so only genuine in-context copying scores well).

**What's in `log.h5`**
- `eval_iter`: shape `(n_evals,)`, the iteration of each evaluation
- `<evaluator>/<metric>`: shape `(n_evals, eval_iters)`, one value **per evaluation sequence**
- `train_loss`, `train_iter`, `train_grad_norm`: per training step

**Units:** iterations in this codebase are *sequences seen*, not optimiser steps (steps = iterations / `train_bs`).

**Definitions used below** (as fractions of the rise from the first eval to the peak):
- **Onset**: accuracy has covered `ONSET_FRAC` of its total rise
- **Plateau**: accuracy reaches `PLATEAU_FRAC` of its rise *and never drops below it again*

The suggested per-generation training cap at the end is the plateau × 1.5–2. The margin is so that later collapse generations whose induction head forms *later* are not cut off.

## 1. Settings

In [ ]:
# One or more run folders (each containing log.h5, config.json, checkpoints/).
# Add several seeds to check how much the phase-change timing varies.
RUN_PATHS = [
    './ih_paper_reprod/main_paper_is5_ih3_pt2/omniglot50_rl5',
]

EVALUATOR    = 'fsl_val_rl'   # evaluator to investigate
METRIC       = 'acc'          # metric used to locate the phase change
ONSET_FRAC   = 0.1            # fraction of the rise that marks onset
PLATEAU_FRAC = 0.9            # fraction of the rise that marks plateau
SMOOTH       = 3              # moving-average window (in evals) before detection
OUT_DIR      = 'investigate_out'

## 2. Imports and helpers

In [ ]:
import json
import os
import re

import h5py
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

# Metrics written by eval_step in main.py
EVAL_METRICS = ['acc', 'in_context_acc', 'out_context_acc', 'loss', 'prob',
                'in_context_prob', 'out_context_prob', 'use_context_prob']

In [ ]:
def resolve_paths(path):
    # Accept a run folder or a log.h5 path; return (run_folder, log_path).
    if os.path.isdir(path):
        return path, os.path.join(path, 'log.h5')
    return os.path.dirname(os.path.abspath(path)), path


def load_run(path):
    # Load every evaluator, the per-sequence values, training curves, config and checkpoint list.
    run_folder, log_path = resolve_paths(path)
    if not os.path.exists(log_path):
        raise FileNotFoundError(f'No log.h5 found at {log_path}')

    run = {'name': os.path.basename(os.path.normpath(run_folder)),
           'run_folder': run_folder, 'per_seq': {}, 'train': {}}
    with h5py.File(log_path, 'r') as f:
        run['evaluators'] = sorted(k for k in f.keys() if isinstance(f[k], h5py.Group))
        eval_iter = np.array(f['eval_iter'])
        for ev in run['evaluators']:
            run['per_seq'][ev] = {m: np.array(f[ev][m]) for m in f[ev].keys()}  # (n_evals, eval_iters)
        for k in ['train_iter', 'train_loss', 'train_grad_norm']:
            if k in f:
                run['train'][k] = np.array(f[k])

    # A run that stopped mid-eval can leave eval_iter and the metric arrays
    # with different lengths; trim everything to the shortest.
    n = min([len(eval_iter)] + [len(v) for ev in run['per_seq'].values() for v in ev.values()])
    run['eval_iter'] = eval_iter[:n]
    run['per_seq'] = {ev: {m: v[:n] for m, v in d.items()} for ev, d in run['per_seq'].items()}
    run['mean'] = {ev: {m: v.mean(axis=1) for m, v in d.items()} for ev, d in run['per_seq'].items()}

    cfg_path = os.path.join(run_folder, 'config.json')
    run['config'] = json.load(open(cfg_path)) if os.path.exists(cfg_path) else {}

    ckpt_dir = os.path.join(run_folder, 'checkpoints')
    ckpts = []
    if os.path.isdir(ckpt_dir):
        for fn in os.listdir(ckpt_dir):
            m = re.fullmatch(r'(\d+)\.eqx', fn)
            if m:
                ckpts.append(int(m.group(1)))
    run['ckpts'] = np.array(sorted(ckpts))
    return run


def smooth(y, window):
    if window <= 1 or len(y) < window:
        return y
    pad = window // 2
    ypad = np.pad(y, (pad, window - 1 - pad), mode='edge')
    return np.convolve(ypad, np.ones(window) / window, mode='valid')


def find_phase_change(iters, y, onset_frac=ONSET_FRAC, plateau_frac=PLATEAU_FRAC, window=SMOOTH):
    # Onset/plateau as fractions of the rise from the first eval to the (smoothed) peak.
    s = smooth(y, window)
    start, peak_idx = s[0], int(np.argmax(s))
    peak = s[peak_idx]
    rise = peak - start
    res = {'start': float(y[0]), 'peak': float(y[peak_idx]), 'peak_iter': int(iters[peak_idx]),
           'final': float(y[-1]), 'rise': float(rise), 'onset_iter': None, 'plateau_iter': None,
           'max_drop_after_peak': float(peak - s[peak_idx:].min())}
    if rise < 0.05:
        return res  # no meaningful phase change

    above = np.where(s >= start + onset_frac * rise)[0]
    if len(above):
        res['onset_iter'] = int(iters[above[0]])
    below = np.where(s < start + plateau_frac * rise)[0]
    last_below = below[-1] if len(below) else -1
    if last_below + 1 < len(s):
        res['plateau_iter'] = int(iters[last_below + 1])
    return res


def nearest_ckpt(ckpts, it):
    if it is None or len(ckpts) == 0:
        return None
    return int(ckpts[np.argmin(np.abs(ckpts - it))])

## 3. Load the runs

In [ ]:
runs = [load_run(p) for p in RUN_PATHS]

for r in runs:
    cfg = r['config']
    print(f"Run: {r['name']}")
    print(f"  lr={cfg.get('lr')}  train_iters={cfg.get('train_iters')}  train_bs={cfg.get('train_bs')}  "
          f"init_seed={cfg.get('init_seed')}  d_model={cfg.get('d_model')}")
    print(f"  evaluators: {r['evaluators']}")
    print(f"  {len(r['eval_iter'])} evals, iterations {r['eval_iter'][0]:,} .. {r['eval_iter'][-1]:,}")
    print(f"  {len(r['ckpts'])} checkpoints")
    assert EVALUATOR in r['per_seq'], f'{EVALUATOR} not in this run'
    assert METRIC in r['per_seq'][EVALUATOR], f"{METRIC} not in {sorted(r['per_seq'][EVALUATOR])}"

## 4. Locate the phase change

In [ ]:
results = []
for r in runs:
    pc = find_phase_change(r['eval_iter'], r['mean'][EVALUATOR][METRIC])
    results.append(pc)
    bs = r['config'].get('train_bs')
    fmt = lambda it: 'not reached' if it is None else f'{it:,}' + (f' (~{it // bs:,} steps)' if bs else '')

    print('=' * 70)
    print(f"{r['name']}  |  {EVALUATOR}/{METRIC}")
    print(f"  start {pc['start']:.3f}   peak {pc['peak']:.3f} (iter {pc['peak_iter']:,})   final {pc['final']:.3f}")
    if pc['onset_iter'] is None:
        print(f"  No phase change detected (rise {pc['rise']:.3f}). Train longer or check the run.")
        continue
    print(f'  Onset   : {fmt(pc["onset_iter"])}')
    print(f'  Plateau : {fmt(pc["plateau_iter"])}')
    if pc['plateau_iter'] is not None:
        print(f"  Phase change lasts {pc['plateau_iter'] - pc['onset_iter']:,} iterations")
    if pc['max_drop_after_peak'] > 0.05:
        print(f"  WARNING: accuracy falls by {pc['max_drop_after_peak']:.3f} after its peak (instability / transience?)")
    print(f"  Nearest checkpoints: onset -> {nearest_ckpt(r['ckpts'], pc['onset_iter'])}, "
          f"plateau -> {nearest_ckpt(r['ckpts'], pc['plateau_iter'])}")

### Every metric at onset, plateau and end of training
Useful sanity checks:
- `in_context_acc` should reach ~1 at the plateau (the model picks the right one of the two context labels).
- `use_context_prob` measures how much probability goes to labels that appear in context at all. It should approach 1 once the model relies on the context.

In [ ]:
for r, pc in zip(runs, results):
    idx = {}
    for lab in ['onset', 'plateau']:
        it = pc[f'{lab}_iter']
        idx[lab] = int(np.searchsorted(r['eval_iter'], it)) if it is not None else None
    idx['final'] = len(r['eval_iter']) - 1

    print(f"{r['name']}  ({EVALUATOR})")
    print(f"  {'metric':<18}{'onset':>10}{'plateau':>10}{'final':>10}")
    for m in [m for m in EVAL_METRICS if m in r['mean'][EVALUATOR]]:
        vals = [f"{r['mean'][EVALUATOR][m][i]:>10.3f}" if i is not None else f"{'-':>10}"
                for i in (idx['onset'], idx['plateau'], idx['final'])]
        print(f'  {m:<18}' + ''.join(vals))
    print()

## 5. Training curves around the phase change

In [ ]:
panels = [(METRIC, f'{METRIC} ({EVALUATOR})'),
          ('in_context_acc', 'in-context acc'),
          ('use_context_prob', 'prob. mass on in-context labels'),
          ('loss', 'eval loss')]
panels = [p for p in panels if any(p[0] in r['mean'][EVALUATOR] for r in runs)]

fig, axs = plt.subplots(1, len(panels) + 1, figsize=(4.2 * (len(panels) + 1), 3.4))
for r, pc in zip(runs, results):
    for ax, (m, title) in zip(axs, panels):
        if m not in r['mean'][EVALUATOR]:
            continue
        line, = ax.plot(r['eval_iter'], r['mean'][EVALUATOR][m], lw=1.5, label=r['name'])
        ax.set_title(title, fontsize=10)
        for key, ls in [('onset_iter', ':'), ('plateau_iter', '--')]:
            if pc[key] is not None:
                ax.axvline(pc[key], color=line.get_color(), ls=ls, lw=1)
    if 'train_loss' in r['train']:
        tl = r['train']['train_loss']
        ti = r['train'].get('train_iter')
        x = ti if ti is not None and len(ti) == len(tl) else np.arange(len(tl))
        axs[-1].plot(x, smooth(tl, max(1, len(tl) // 200)), lw=1)
        axs[-1].set_title('train loss (smoothed)', fontsize=10)

for ax in axs:
    ax.set_xlabel('sequences seen')
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
    ax.grid(alpha=0.3)
if len(runs) > 1:
    axs[0].legend(fontsize=8)
fig.suptitle('dotted = onset, dashed = plateau', fontsize=9)
fig.tight_layout()
os.makedirs(OUT_DIR, exist_ok=True)
fig.savefig(os.path.join(OUT_DIR, f'phase_change_{EVALUATOR}_{METRIC}.png'), dpi=150)
plt.show()

## 6. Compare evaluators
If the model is genuinely doing in-context learning, all evaluators should jump at about the same time:
- `fsl_train`: label pairs seen in training
- `fsl_val_rl`: held-out label pairs
- `fsl_train_valex`: unseen exemplars of training classes
- `fsl_test_class`: unseen classes

If `fsl_train` rises well before `fsl_val_rl`, the model is partly memorising label pairs.

In [ ]:
fig, axs = plt.subplots(1, len(runs), figsize=(5 * len(runs), 3.4), squeeze=False)
for ax, r in zip(axs[0], runs):
    for ev in r['evaluators']:
        if METRIC in r['mean'][ev]:
            ax.plot(r['eval_iter'], r['mean'][ev][METRIC], lw=1.3, label=ev)
            pc_ev = find_phase_change(r['eval_iter'], r['mean'][ev][METRIC])
            print(f"{r['name']:<30} {ev:<18} onset {pc_ev['onset_iter']}  plateau {pc_ev['plateau_iter']}")
    ax.set_title(r['name'], fontsize=10)
    ax.set_xlabel('sequences seen')
    ax.set_ylabel(METRIC)
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f'evaluators_{METRIC}.png'), dpi=150)
plt.show()

## 7. Per-sequence view: do all sequences switch at once?
`log.h5` stores one value per evaluation sequence. For each `fsl_val_rl` sequence we find the first eval where the model puts more than 50% probability on the correct label, then plot when that happens.
- A narrow spike means the whole circuit switches on together, the classic phase change.
- A wide spread, or a group that never switches, is worth reporting. In the collapse generations, compare this distribution with generation 0.

In [ ]:
THRESH = 0.5
fig, axs = plt.subplots(1, len(runs), figsize=(5 * len(runs), 3.2), squeeze=False)
for ax, r, pc in zip(axs[0], runs, results):
    prob = r['per_seq'][EVALUATOR].get('prob')
    if prob is None:
        ax.set_title('no per-sequence prob logged')
        continue
    solved = prob > THRESH                                    # (n_evals, n_seqs)
    ever = solved.any(axis=0)
    first = np.where(ever, r['eval_iter'][np.argmax(solved, axis=0)], np.nan)
    ax.hist(first[ever], bins=40)
    if pc['onset_iter'] is not None:
        ax.axvline(pc['onset_iter'], ls=':', c='k', lw=1)
    if pc['plateau_iter'] is not None:
        ax.axvline(pc['plateau_iter'], ls='--', c='k', lw=1)
    ax.set_title(f"{r['name']}: {ever.mean():.1%} of sequences ever solved", fontsize=10)
    ax.set_xlabel(f'first iteration with p(correct) > {THRESH}')
    ax.ticklabel_format(axis='x', style='sci', scilimits=(0, 0))
    ax.grid(alpha=0.3)
    q = np.nanpercentile(first, [10, 50, 90])
    print(f"{r['name']}: 10/50/90th percentile switch iteration = {q[0]:,.0f} / {q[1]:,.0f} / {q[2]:,.0f}")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, f'per_sequence_switch_{EVALUATOR}.png'), dpi=150)
plt.show()

## 8. Suggested per-generation training cap

In [ ]:
plateaus = [pc['plateau_iter'] for pc in results if pc['plateau_iter'] is not None]
if not plateaus:
    print('No plateau found in any run; cannot suggest a cap.')
else:
    latest = max(plateaus)
    print(f'Plateau iterations across runs: {[f"{p:,}" for p in plateaus]}')
    print(f'Suggested cap: {int(1.5 * latest):,} (1.5x latest plateau) to {2 * latest:,} (2x)')
    bs = runs[0]['config'].get('train_bs')
    if bs:
        print(f'  = {int(1.5 * latest) // bs:,} to {2 * latest // bs:,} optimiser steps at batch size {bs}')

summary = {r['name']: pc for r, pc in zip(runs, results)}
with open(os.path.join(OUT_DIR, 'summary.json'), 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Saved summary to {os.path.join(OUT_DIR, 'summary.json')}")